In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_faq = pd.read_csv('faq-data/df_faq.csv', index_col=0)
df_faq['Question'] = '(' + df_faq['Type'] + ') ' + df_faq['Question']
df_faq['Answer'] = '(' + df_faq['Type'] + ') ' + df_faq['Answer']

# Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer

questions = df_faq['Question']
answers = df_faq['Answer']

In [ ]:
sentence_transformer = SentenceTransformer('bert-base-nli-mean-tokens')
question_embeddings = sentence_transformer.encode(questions)
question_embeddings.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_qQ_similarities(query):
    query_embedding = sentence_transformer.encode([query])

    similarities = cosine_similarity(
        question_embeddings, query_embedding
    )

    df = pd.DataFrame(similarities, columns = ['Similarity'])
    df['Question'] = questions
    df = df.sort_values(by='Similarity', ascending=False)

    return df

In [ ]:
get_qQ_similarities('When is the application deadline for scholarships?')

In [ ]:
answer_embeddings = sentence_transformer.encode(answers)
answer_embeddings.shape

In [ ]:
def get_qA_similarities(query):
    query_embedding = sentence_transformer.encode([query])

    similarities = cosine_similarity(
        answer_embeddings, query_embedding
    )

    df = pd.DataFrame(similarities, columns = ['Similarity'])
    df['Answer'] = answers
    df = df.sort_values(by='Similarity', ascending=False)

    return df

In [ ]:
get_qA_similarities('How much does the programme cost?')

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# create new column representing relevance between each QA pair
df_faq['QA Relevance'] = 1

In [ ]:
# generate 24 negative samples for each QA pair
import random

def generate_negative_samples(df):
    questions = df['Question']
    answers = df['Answer']

    df_negative = pd.DataFrame()
    for i, q in enumerate(questions):
        remaining_answers = answers.drop(index=i)
        df_negative_samples = pd.DataFrame(random.sample(list(remaining_answers), 2), columns=['Answer'])
        df_negative_samples['Question'] = q
        df_negative_samples['QA Relevance'] = 0
        df_negative = df_negative.append(df_negative_samples)
    
    return df_negative

In [ ]:
# generate the final dataset with negative samples included

df_negative = generate_negative_samples(df_faq)

# df_faq = df_faq.drop(columns=['Type'])
df_final = df_faq.append(df_negative).reset_index(drop = True)

# rename the columns to the required 'text_a', 'text_b', 'labels'
df_final = df_final.rename(columns={'Question': 'text_a', 'Answer': 'text_b', 'QA Relevance': 'labels'})

In [ ]:
df_final.to_csv('faq-data/df_final.csv')

In [ ]:
df_final = pd.read_csv('data/df_final.csv', index_col=0)

In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df_final, test_size=0.2, random_state=7600, shuffle=True)

In [ ]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import logging

In [ ]:
logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger('transformers')
transformers_logger.setLevel(logging.WARNING)

In [ ]:
import torch
cuda_available = torch.cuda.is_available()

**Trial 1: ROBERTA using 24 negative samples per positive sample (same as in the paper)**
- N_EPOCHS = 5

In [ ]:
# define model configuration parameters
N_EPOCHS = 5

# configure the classification model
bert_qA_args = ClassificationArgs(num_train_epochs=N_EPOCHS)

In [ ]:
# define the model
bert_qA = ClassificationModel('roberta', 'roberta-base', use_cuda=cuda_available, args=bert_qA_args)

In [ ]:
bert_qA.train_model(df_train)

In [ ]:
t1_result, t1_model_outputs, t1_wrong_predictions = bert_qA.eval_model(
    df_test
)

In [ ]:
qA_test_samples = [
    [
        'What is my application deadline?', 
        'You may wish to go through the Important Dates when applying to the University. Applications submitted after the first round application deadline will be considered on a rolling basis subject to programme availability.'
    ],

    [
        'When will I know the result of my application?',
        'You may wish to go through the Important Dates when applying to the University. Applications submitted after the first round application deadline will be considered on a rolling basis subject to programme availability.'
    ],

    [
        'Hi',
        'You may wish to go through the Important Dates when applying to the University. Applications submitted after the first round application deadline will be considered on a rolling basis subject to programme availability.'
    ]
]

In [ ]:
t1_predictions, t1_raw_outputs = bert_qA.predict(qA_test_samples)

In [ ]:
t1_predictions

**Trial 2: RoBERTA with 2 negative samples per positive sample**
- N_EPOCHS = 7

In [ ]:
# configure the classification model
N_EPOCHS = 7
roberta_qA_args = ClassificationArgs(num_train_epochs=N_EPOCHS)

In [ ]:
roberta_qA = ClassificationModel('roberta', 'roberta-base', use_cuda=cuda_available, args=roberta_qA_args)

In [ ]:
roberta_qA.train_model(df_train, output_dir='outputs/roberta-t2')

In [ ]:
t2_result, t2_model_outputs, t2_wrong_predictions = roberta_qA.eval_model(
    df_test
)

In [ ]:
t2_predictions, t2_raw_outputs = roberta_qA.predict(qA_test_samples)

In [ ]:
# test some tricky sentences
query = 'I wanna know what the application deadline is.'
available_answers = df_faq['Answer']

qa_test_samples_tricky = []
for a in available_answers:
    qa_test_samples_tricky.append([query, a])

# get the predictions
t2_predictions_tricky, t2_raw_outputs_tricky = roberta_qA.predict(qa_test_samples_tricky)

# format the predictions into an easy-to-read dataframe
df_t2_predictions_tricky = pd.DataFrame(qa_test_samples_tricky, columns=['User Query', 'Answer in Database'])
df_t2_predictions_tricky['Question in Database'] = df_faq['Question']
df_t2_predictions_tricky['Predicted QA Relevance'] = t2_predictions_tricky

In [ ]:
df_t2_predictions_tricky.to_csv('outputs/roberta-t2/df_t2_predictions_tricky.csv')

**Trial 3: ROBERTA initially trained on SQuAD (but not for classification purposes)**
- n_epochs = 12

In [ ]:
# configure the classification model
N_EPOCHS = 12
roberta_qA_squad2_args = ClassificationArgs(num_train_epochs=N_EPOCHS)

In [ ]:
roberta_qA_squad2 = ClassificationModel('roberta', 'deepset/roberta-base-squad2', use_cuda=cuda_available, args=roberta_qA_squad2_args)

In [ ]:
roberta_qA_squad2.train_model(df_train, output_dir='outputs/roberta-t3-squad2')

In [ ]:
t3_result, t3_model_outputs, t3_wrong_predictions = roberta_qA_squad2.eval_model(
    df_test
)

In [ ]:
t3_predictions, t3_raw_outputs = roberta_qA_squad2.predict(qA_test_samples)

In [ ]:
t3_predictions

**Trial 4**

In [ ]:
# configure the classification model
N_EPOCHS = 20
mpnet_qA_args = ClassificationArgs(num_train_epochs=N_EPOCHS)

In [ ]:
mpnet_qA = ClassificationModel('mpnet', 'sentence-transformers/all-mpnet-base-v2', use_cuda=cuda_available, args=mpnet_qA_args)

In [ ]:
mpnet_qA.train_model(df_train, output_dir='outputs/mpnet-t4')

In [ ]:
t4_result, t4_model_outputs, t4_wrong_predictions = mpnet_qA.eval_model(
    df_test
)

In [ ]:
t4_predictions, t4_raw_outputs = mpnet_qA.predict(qA_test_samples)

In [ ]:
t4_predictions

In [ ]:
predictions_tricky

In [ ]:
# test some tricky sentences
query = 'I wanna know what the application deadline is.'
available_answers = df_faq['Answer']

qa_test_samples_tricky = []
for a in available_answers:
    qa_test_samples_tricky.append([query, a])

# get the predictions
t4_predictions_tricky, t4_raw_outputs_tricky = mpnet_qA.predict(qa_test_samples_tricky)

# format the predictions into an easy-to-read dataframe
predictions_tricky = pd.DataFrame(qa_test_samples_tricky, columns=['User Query', 'Answer in Database'])
predictions_tricky['Question in Database'] = df_faq['Question']
predictions_tricky['Predicted QA Relevance'] = t4_predictions_tricky

In [ ]:
predictions_tricky.sort_values(by='Predicted QA Relevance', ascending=False).to_csv('outputs/mpnet-t4/df_t4_predictions_tricky.csv')

**Trial 5: BERT**

In [ ]:
# configure the classification model
N_EPOCHS = 10
bert_qA_args = ClassificationArgs(num_train_epochs=N_EPOCHS)

In [ ]:
bert_qA = ClassificationModel('bert', 'bert-base-uncased', use_cuda=cuda_available, args=bert_qA_args)

In [ ]:
bert_qA.train_model(df_train, output_dir='outputs/bert-t5')

In [ ]:
t5_result, t5_model_outputs, t5_wrong_predictions = bert_qA.eval_model(
    df_test
)

In [ ]:
t5_predictions, t5_raw_outputs = bert_qA.predict(qA_test_samples)

In [ ]:
t5_predictions

In [ ]:
# configure the classification model
N_EPOCHS = 7
bert_qA_pretrained_args = ClassificationArgs(num_train_epochs=N_EPOCHS)

In [ ]:
bert_qA_pretrained = ClassificationModel('roberta', 'data/uncased_L-12_H-768_A-12', use_cuda=cuda_available, args=bert_qA_pretrained_args)

In [ ]:
roberta_qA_squad2.train_model(df_train, output_dir='outputs/roberta-t3-squad2')

In [ ]:
t3_result, t3_model_outputs, t3_wrong_predictions = roberta_qA_squad2.eval_model(
    df_test
)

In [ ]:
t3_predictions, t3_raw_outputs = roberta_qA_squad2.predict(qA_test_samples)

In [ ]:
t3_predictions